In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#teacher mlp
input_dim = 32
batch_size = 256
val_size = 16384
torch.manual_seed(4948497)
teacher = nn.Sequential(nn.Linear(input_dim, 256), nn.GELU(), nn.Linear(256,256), nn.GELU(), nn.Linear(256,1))

torch.manual_seed(4948492 + 1)  # eigener Seed für die Probe, unabhängig vom Teacher-Seed
with torch.no_grad():
    probe = torch.randn(65536, input_dim)
    out = teacher(probe)
out_mean = out.mean().item()
out_scale = 1.0 / out.std().item()

def clean_target(x):
    with torch.no_grad():
        return (teacher(x) - out_mean) * out_scale
    
def cosine_lr_lambda(total_steps: int, warmup_frac: float = 0.08,
                     start_factor: float = 0.01, end_factor: float = 0.1):
    """LR multiplier: linear warmup eta/100 -> eta over ``warmup_frac`` of the steps,
    then cosine annealing eta -> eta/10. The per-cell cosine schedule used throughout."""
    warmup_steps = max(1, int(warmup_frac * total_steps))

    def fn(step: int) -> float:
        if step < warmup_steps:
            return start_factor + (1.0 - start_factor) * step / warmup_steps
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return end_factor + (1.0 - end_factor) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return fn

In [ ]:
def make_std():
    torch.manual_seed(384929)
    return nn.Sequential(nn.Linear(input_dim, 64), nn.GELU(), nn.Linear(64,64), nn.GELU(), nn.Linear(64,1))

def train_student(lr, n_train_steps):
    mlp = make_std()
    loss_function = nn.MSELoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_lr_lambda(n_train_steps))
    
    for i in range(n_train_steps):
        x_input = torch.randn(batch_size, input_dim) # in oder outside of loop?
        with torch.no_grad():
            target = clean_target(x_input)

        optimizer.zero_grad()
        loss = loss_function(mlp(x_input),target)
        loss.backward()
        optimizer.step()
        scheduler.step()

    gv = torch.Generator().manual_seed(496302)
    x_val = torch.randn(val_size, input_dim, generator=gv)
    with torch.no_grad():
        y_val = clean_target(x_val)

    val_loss = loss_function(mlp(x_val), y_val)
    return val_loss


In [ ]:
#evaluation of val_loss for different lr
learn_rates = np.geomspace(3e-4, 1e-1, 9)
losses = []
for lr in learn_rates:
    val_loss = train_student(lr, 2048).item()
    losses.append(val_loss)
    

In [ ]:
def plot_lr_tuning(lrs, losses):
    """L1 figure: validation loss vs learning rate at one cell, with the parabola fit
    (in log10(LR)) and its vertex eta*."""
    lrs = np.asarray(lrs, dtype=float)
    x = np.log10(lrs)
    a, b, c = np.polyfit(x, losses, 2)
    xg = np.linspace(x.min(), x.max(), 100)
    fig, ax = plt.subplots(figsize=(6, 4.2))
    ax.plot(10 ** xg, a * xg ** 2 + b * xg + c, "-", color="0.6", lw=1.5, label="parabola fit")
    ax.plot(lrs, losses, "o", color="steelblue", ms=7, label="measured")
    ax.set(xscale="log", xlabel=r"learning rate $\eta$",ylabel="validation loss")
    ax.legend(frameon=False)
    fig.tight_layout()
    

plot_lr_tuning(learn_rates, losses)

